In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
import os
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/SVS-Cyber')
os.environ['SVS_CYBER_BASE'] = str(DRIVE_BASE)
DATA = DRIVE_BASE / 'data'
RAW = DATA / 'raw'
ACQUIRED = DATA / 'acquired'

for folder in [RAW, ACQUIRED]:
    folder.mkdir(parents=True, exist_ok=True)

print('Drive base:', DRIVE_BASE)
print('Raw data:', RAW)
print('Acquired data:', ACQUIRED)


In [ ]:
%pip install -q requests gzip tqdm


In [ ]:
import json, gzip, os, re, time, urllib.request, urllib.parse
from pathlib import Path
from tqdm.notebook import tqdm

def download(url, dest, timeout=60):
    req = urllib.request.Request(url, headers={'User-Agent': 'SVS-Cyber-Research/0.1'})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        dest.parent.mkdir(parents=True, exist_ok=True)
        total = int(r.headers.get('Content-Length', 0))
        with dest.open('wb') as f, tqdm(
            total=total,
            unit='B',
            unit_scale=True,
            desc=dest.name,
        ) as pbar:
            while True:
                chunk = r.read(1024*1024)
                if not chunk:
                    break
                f.write(chunk)
                pbar.update(len(chunk))

def append_jsonl(path, records):
    with path.open('a', encoding='utf-8') as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=True) + '\n')

print('Download utilities with progress bars ready.')


In [ ]:
# NLP / General chat datasets
print('=== Downloading OpenAssistant OASST1 ===')
url = 'https://huggingface.co/datasets/OpenAssistant/oasst1/resolve/main/2023-04-12_oasst_ready.messages.jsonl.gz'
dest = RAW / 'openassistant_oasst1_messages.jsonl.gz'
if not dest.exists():
    download(url, dest)
    print('Downloaded OASST1:', dest.stat().st_size / 1024 / 1024, 'MB')
else:
    print('OASST1 already exists')

print('=== Downloading OpenAssistant OASST2 ===')
url = 'https://huggingface.co/datasets/OpenAssistant/oasst2/resolve/main/2023-11-05_oasst2_ready.messages.jsonl.gz'
dest = RAW / 'openassistant_oasst2_messages.jsonl.gz'
if not dest.exists():
    download(url, dest)
    print('Downloaded OASST2:', dest.stat().st_size / 1024 / 1024, 'MB')
else:
    print('OASST2 already exists')

print('=== Downloading Databricks Dolly 15k ===')
url = 'https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl'
dest = RAW / 'databricks_dolly_15k.jsonl'
if not dest.exists():
    download(url, dest)
    print('Downloaded Dolly:', dest.stat().st_size / 1024 / 1024, 'MB')
else:
    print('Dolly already exists')


In [ ]:
# Cybersecurity datasets
print('=== Downloading MalwareBazaar metadata ===')
AUTH_KEY = 'b2f7942d27dab1189dee0aa1b0def658a5ed47c8164639cf'
mb_url = 'https://mb-api.abuse.ch/api/v1/'
seen = set()
records = []
pbar = tqdm(total=7000, desc='MalwareBazaar records')
while len(records) < 7000:
    payload = urllib.parse.urlencode({'query': 'recent_detections', 'limit': '500'}).encode()
    req = urllib.request.Request(mb_url, data=payload, headers={
        'Content-Type': 'application/x-www-form-urlencoded',
        'Auth-Key': AUTH_KEY,
        'User-Agent': 'SVS-Cyber-Research/0.1'
    })
    with urllib.request.urlopen(req, timeout=30) as r:
        resp = json.loads(r.read().decode('utf-8'))
        if resp.get('query_status') != 'ok':
            print('API error:', resp)
            break
        batch = resp.get('data', [])
        if not batch:
            break
        new_count = 0
        for rec in batch:
            sha = rec.get('sha256_hash') or rec.get('md5') or rec.get('sha1_hash')
            if not sha or sha in seen:
                continue
            seen.add(sha)
            records.append(rec)
            new_count += 1
            if len(records) >= 7000:
                break
        pbar.update(new_count)
        print(f'  Fetched {len(records)} total records')
        time.sleep(1)
pbar.close()

mb_out = RAW / 'malwarebazaar_meta.jsonl'
with mb_out.open('w', encoding='utf-8') as f:
    for rec in records[:7000]:
        f.write(json.dumps(rec, ensure_ascii=True) + '\n')
print('Saved MalwareBazaar:', len(records[:7000]), 'records,', mb_out.stat().st_size / 1024, 'KB')

print('=== Downloading SOREL-20M meta.db ===')
sorel_url = 'https://sorel-20m.s3.amazonaws.com/09-DEC-2020/processed-data/meta.db'
sorel_dest = RAW / 'sorel20m' / 'meta.db'
if not sorel_dest.exists():
    download(sorel_url, sorel_dest)
    print('Downloaded SOREL-20M meta.db:', sorel_dest.stat().st_size / 1024 / 1024 / 1024, 'GB')
else:
    print('SOREL-20M meta.db already exists')

print('=== Downloading CISA KEV ===')
cisa_url = 'https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json'
cisa_dest = RAW / 'cisa_kev.json'
if not cisa_dest.exists():
    download(cisa_url, cisa_dest)
    print('Downloaded CISA KEV:', cisa_dest.stat().st_size / 1024, 'KB')
else:
    print('CISA KEV already exists')


In [ ]:
# Normalize SOREL-20M meta.db to JSONL
print('=== Normalizing SOREL-20M to JSONL ===')
import sqlite3
sorel_jsonl = ACQUIRED / 'sorel20m_meta.jsonl'
if sorel_dest.exists() and not sorel_jsonl.exists():
    conn = sqlite3.connect(str(sorel_dest))
    cursor = conn.cursor()
    cursor.execute('PRAGMA table_info(meta)')
    columns = [col[1] for col in cursor.fetchall()]
    cursor.execute('SELECT * FROM meta LIMIT 50000')
    rows = cursor.fetchall()
    conn.close()
    with sorel_jsonl.open('w', encoding='utf-8') as f:
        for row in tqdm(rows, desc='SOREL-20M rows'):
            data = dict(zip(columns, row))
            record = {
                'record_id': f'sorel20m-{data.get(\"sha256\", \"\")[:16]}',
                'source': 'sorel20m',
                'retrieved_at': '2026-08-28',
                'content': {
                    'dataset': 'SOREL-20M',
                    'sha256': data.get('sha256'),
                    'is_malware': data.get('is_malware'),
                    'labels': {col: data.get(col) for col in columns if col != 'sha256' and data.get(col) is not None},
                }
            }
            f.write(json.dumps(record, ensure_ascii=True) + '\n')
    print('Normalized SOREL-20M:', len(rows), 'records ->', sorel_jsonl)
elif sorel_jsonl.exists():
    print('SOREL-20M JSONL already exists')
else:
    print('SOREL-20M meta.db not found, skipping normalization')


In [ ]:
# Normalize all datasets into unified corpus
print('=== Normalizing all datasets ===')

def normalize_record(record_id, source, retrieved_at, content):
    return {
        'record_id': record_id,
        'source': source,
        'retrieved_at': retrieved_at,
        'content': content,
    }

normalized = []

# Normalize OpenAssistant OASST1
oasst1_path = RAW / 'openassistant_oasst1_messages.jsonl.gz'
if oasst1_path.exists():
    with gzip.open(oasst1_path, 'rt', encoding='utf-8') as f:
        lines = f.readlines()
    for line in tqdm(lines, desc='OASST1'):
        line = line.strip()
        if not line:
            continue
        data = json.loads(line)
        normalized.append(normalize_record(
            f'oasst1-{data.get(\"message_id\", i)}',
            'openassistant_oasst1',
            '2026-08-28',
            {
                'message_id': data.get('message_id'),
                'parent_id': data.get('parent_id'),
                'user_id': data.get('user_id'),
                'role': data.get('role'),
                'text': (data.get('text', '') or '')[:2000],
                'lang': data.get('lang'),
                'tree_id': data.get('message_tree_id'),
                'rank': data.get('rank'),
                'synthetic': data.get('synthetic'),
                'model_name': data.get('model_name'),
            }
        ))
    print('Normalized OASST1:', len(normalized), 'records')

# Normalize OpenAssistant OASST2
oasst2_path = RAW / 'openassistant_oasst2_messages.jsonl.gz'
if oasst2_path.exists():
    with gzip.open(oasst2_path, 'rt', encoding='utf-8') as f:
        lines = f.readlines()
    for line in tqdm(lines, desc='OASST2'):
        line = line.strip()
        if not line:
            continue
        data = json.loads(line)
        normalized.append(normalize_record(
            f'oasst2-{data.get(\"message_id\", i)}',
            'openassistant_oasst2',
            '2026-08-28',
            {
                'message_id': data.get('message_id'),
                'parent_id': data.get('parent_id'),
                'user_id': data.get('user_id'),
                'role': data.get('role'),
                'text': (data.get('text', '') or '')[:2000],
                'lang': data.get('lang'),
                'tree_id': data.get('message_tree_id'),
                'rank': data.get('rank'),
                'synthetic': data.get('synthetic'),
                'model_name': data.get('model_name'),
            }
        ))
    print('Normalized OASST2:', len(normalized), 'records')

# Normalize Dolly
dolly_path = RAW / 'databricks_dolly_15k.jsonl'
if dolly_path.exists():
    with dolly_path.open('r', encoding='utf-8') as f:
        lines = f.readlines()
    for i, line in enumerate(tqdm(lines, desc='Dolly')):
        line = line.strip()
        if not line:
            continue
        data = json.loads(line)
        normalized.append(normalize_record(
            f'dolly-{i:06d}',
            'dolly',
            '2026-08-28',
            {
                'category': data.get('category'),
                'instruction': (data.get('instruction', '') or '')[:2000],
                'context': (data.get('context', '') or '')[:2000],
                'response': (data.get('response', '') or '')[:2000],
            }
        ))
    print('Normalized Dolly:', len(normalized), 'records')

# Normalize MalwareBazaar
mb_path = RAW / 'malwarebazaar_meta.jsonl'
if mb_path.exists():
    with mb_path.open('r', encoding='utf-8') as f:
        lines = f.readlines()
    for line in tqdm(lines, desc='MalwareBazaar'):
        line = line.strip()
        if not line:
            continue
        data = json.loads(line)
        normalized.append(normalize_record(
            f'malwarebazaar-{data.get(\"sha256_hash\", \"\")[:16]}',
            'malwarebazaar',
            '2026-08-28',
            {
                'sha256': data.get('sha256_hash'),
                'sha3_384': data.get('sha3_384_hash'),
                'sha1': data.get('sha1_hash'),
                'md5': data.get('md5_hash'),
                'first_seen': data.get('first_seen'),
                'file_name': data.get('file_name'),
                'signature': data.get('signature'),
                'tags': data.get('tags', []),
                'file_type': data.get('file_type'),
                'reporter': data.get('reporter'),
                'delivery_method': data.get('delivery_method'),
            }
        ))
    print('Normalized MalwareBazaar:', len(normalized), 'records')

# Include SOREL-20M JSONL if it exists
sorel_jsonl_path = ACQUIRED / 'sorel20m_meta.jsonl'
if sorel_jsonl_path.exists():
    with sorel_jsonl_path.open('r', encoding='utf-8') as f:
        lines = f.readlines()
    for line in tqdm(lines, desc='SOREL-20M'):
        line = line.strip()
        if line:
            normalized.append(json.loads(line))
    print('Included SOREL-20M:', len([r for r in normalized if r.get('source') == 'sorel20m']), 'records')

# Write normalized corpus
normalized_path = ACQUIRED / 'normalized_corpus_merged.jsonl'
with normalized_path.open('w', encoding='utf-8') as f:
    for rec in tqdm(normalized, desc='Writing corpus'):
        f.write(json.dumps(rec, ensure_ascii=True) + '\n')
print('Wrote normalized corpus:', normalized_path)
print('Total normalized records:', len(normalized))


In [ ]:
# Build training splits
print('=== Building training splits ===')

import random
random.seed(42)

corpus_path = ACQUIRED / 'normalized_corpus_merged.jsonl'
if not corpus_path.exists():
    print('No normalized corpus found. Run normalization cells first.')
else:
    records = []
    with corpus_path.open('r', encoding='utf-8') as f:
        for line in tqdm(f, desc='Loading corpus'):
            line = line.strip()
            if line:
                records.append(json.loads(line))
    
    random.shuffle(records)
    n = len(records)
    train_end = int(n * 0.8)
    val_end = train_end + int((n - train_end) / 2)
    
    train = records[:train_end]
    val = records[train_end:val_end]
    test = records[val_end:]
    
    def write_jsonl(path, recs):
        with path.open('w', encoding='utf-8') as f:
            for rec in tqdm(recs, desc=f'Writing {path.name}'):
                f.write(json.dumps(rec, ensure_ascii=True) + '\n')
    
    write_jsonl(DATA / 'pretraining_train.jsonl', train)
    write_jsonl(DATA / 'pretraining_val.jsonl', val)
    write_jsonl(DATA / 'pretraining_test.jsonl', test)
    
    print(f'pretraining_train: {len(train)} records')
    print(f'pretraining_val: {len(val)} records')
    print(f'pretraining_test: {len(test)} records')
    print('Training splits written to Drive.')


In [ ]:
# Verify outputs
import os

print('=== Verification ===')
for name in ['pretraining_train.jsonl', 'pretraining_val.jsonl', 'pretraining_test.jsonl']:
    path = DATA / name
    if path.exists():
        size_mb = os.path.getsize(path) / 1024 / 1024
        with open(path, 'r', encoding='utf-8') as f:
            count = sum(1 for _ in tqdm(f, desc=f'Counting {name}', unit='lines'))
        print(f'{name}: {count} records, {size_mb:.1f} MB')
    else:
        print(f'{name}: MISSING')

raw_files = list(RAW.glob('*'))
print(f'\nRaw files in Drive: {len(raw_files)}')
for p in raw_files[:10]:
    size_mb = os.path.getsize(p) / 1024 / 1024
    print(f'  {p.name}: {size_mb:.1f} MB')
if len(raw_files) > 10:
    print(f'  ... and {len(raw_files) - 10} more')
